In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings ('ignore')

df = pd.read_csv("Employee.csv")
df.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


# Cleaning Steps:

In [2]:
df.describe()

,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot
count,4653.000000,4653.000000,4653.000000,4653.000000,4653.000000
mean,2015.062970,2.698259,29.393295,2.905652,0.343864
std,1.863377,0.561435,4.826087,1.558240,0.475047
min,2012.000000,1.000000,22.000000,0.000000,0.000000
25%,2013.000000,3.000000,26.000000,2.000000,0.000000
50%,2015.000000,3.000000,28.000000,3.000000,0.000000
75%,2017.000000,3.000000,32.000000,4.000000,1.000000
max,2018.000000,3.000000,41.000000,7.000000,1.000000


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Education                  4653 non-null   str  
 1   JoiningYear                4653 non-null   int64
 2   City                       4653 non-null   str  
 3   PaymentTier                4653 non-null   int64
 4   Age                        4653 non-null   int64
 5   Gender                     4653 non-null   str  
 6   EverBenched                4653 non-null   str  
 7   ExperienceInCurrentDomain  4653 non-null   int64
 8   LeaveOrNot                 4653 non-null   int64
dtypes: int64(5), str(4)
memory usage: 327.3 KB


In [4]:
df = pd.read_csv("Employee.csv")

print('Shape:', df.shape)
print('Null vlues:', df.isnull().sum().sum())
print('Duplicate rows before:', df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print('Duplicate rows after:', df.duplicated().sum())
print('New Shape:', df.shape)

Shape: (4653, 9)
Null vlues: 0
Duplicate rows before: 1889
Duplicate rows after: 0
New Shape: (2764, 9)


# Schema:
## Categorical and Numerical Columns:

## Feature Types
### Numerical:
- JoiningYear  
- Age   
- ExperienceInCurrentDomain  
- PaymentTier  

## Categorical:
- EverBenched  
- Education  
- City  
- Gender  

### Target:
- LeaveOrNot

In [5]:
# Define x nd y:

numerical_features = ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain']
categorical_features = ['Education', 'City', 'Gender', 'EverBenched']

X = df.loc[:, categorical_features + numerical_features].copy()
y = df.loc[:, 'LeaveOrNot'].copy()

print('X shape:', X.shape)
print('y values:', y.value_counts().to_dict())

X shape: (2764, 8)
y values: {0: 1676, 1: 1088}


## Exporting Feature Input Metadata:

In [6]:
def summerize_cat(data, categorical_features):
    results = []
    
    for column in data[categorical_features]:
        # Get the unique name and its unique members to the results list
        members = data[column].unique().tolist()
        # Append the column name and its unique members to the results list
        results.append([column, members])
        
    return pd.DataFrame(results, columns=['Column Name',  'Members'])

# create a DataFrame from the result list
summerize_cat(X, categorical_features)       

,Column Name,Members
0,Education,"[Bachelors, Masters, PHD]"
1,City,"[Bangalore, Pune, New Delhi]"
2,Gender,"[Male, Female]"
3,EverBenched,"[No, Yes]"


### Convert Feature Metadata to Dictionary

In [7]:
summerize_cat(X, categorical_features).to_dict()

{'Column Name': {0: 'Education', 1: 'City', 2: 'Gender', 3: 'EverBenched'},
 'Members': {0: ['Bachelors', 'Masters', 'PHD'],
  1: ['Bangalore', 'Pune', 'New Delhi'],
  2: ['Male', 'Female'],
  3: ['No', 'Yes']}}

### Create Feature Dictionary and Export (my_feature_dict.pkl)

In [8]:
# Create Feature Dictionary
 
def summerize_cat(data, cols):
    return pd.DataFrame({
        'Column Name': cols,
        'Members': [data[c].unique().tolist() for c in cols],
    })

my_feature_dict = {'CATEGORICAL': summerize_cat(X, categorical_features).to_dict(), 'NUMERICAL': {'Column Name': numerical_features}}

In [9]:
# Check my_Features_Dict

my_feature_dict

{'CATEGORICAL': {'Column Name': {0: 'Education',
   1: 'City',
   2: 'Gender',
   3: 'EverBenched'},
  'Members': {0: ['Bachelors', 'Masters', 'PHD'],
   1: ['Bangalore', 'Pune', 'New Delhi'],
   2: ['Male', 'Female'],
   3: ['No', 'Yes']}},
 'NUMERICAL': {'Column Name': ['JoiningYear',
   'PaymentTier',
   'Age',
   'ExperienceInCurrentDomain']}}

# Split:

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
from sklearn.impute import SimpleImputer
#Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train', X_train.shape, 'Test', X_test.shape)

Train (2211, 8) Test (553, 8)


In [11]:
# Check Class Distribution in Train and Test Data:

print('Train Data', '\n', y_train.value_counts(normalize=True), '\n', '\n', 'Test Data',  '\n', y_test.value_counts(normalize=True))

Train Data 
 LeaveOrNot
0    0.607417
1    0.392583
Name: proportion, dtype: float64 
 
 Test Data 
 LeaveOrNot
0    0.60217
1    0.39783
Name: proportion, dtype: float64


**key points:**
- Train: Stay 60.7%, Leave 39.3% 
- Test: Stay 60.2%, Leave 39.8%
- The two ratios are very close, so the test represents the training  data well.

# Creating the Pipeline:

In [12]:
# Numerical Pipeline:(SimpleImputer)

pipeline_num =Pipeline(steps=[
    ('scale_data', MinMaxScaler()),
    ('simple_imputer1', SimpleImputer(strategy='constant', fill_value=0)),
])

In [13]:
# Categorical Pipeline(OneHotEncoder)

Pipeline_cat = Pipeline(steps=[
    ('OneHotEncoder', OneHotEncoder(handle_unknown = 'ignore')),
])

In [14]:
# Preprocessor Stage 1 (pass through)

from sklearn.preprocessing import FunctionTransformer

preprocessor_stage1 = Pipeline(steps=[
    ('pass_through', FunctionTransformer())
])

In [15]:
# Combine Both Pipeline

from sklearn.compose import ColumnTransformer

preprocessor_stage2 = ColumnTransformer(
    transformers=[
        ('cat', Pipeline_cat, categorical_features),   # Categorical Columns
        ('num', pipeline_num, numerical_features),     # Numerical Columns
    ], remainder='drop')

In [16]:
#Preprocessor Stack

preprocessor_stack = Pipeline(steps=[
                               ('preprocessor_stage1', preprocessor_stage1),
                               ('preprocessor_stage2', preprocessor_stage2)
                               ])

### Fitting The Pipeline:

In [17]:
preprocessor_stack.fit(X_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor_stage1', ...), ('preprocessor_stage2', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Education','City','Gender',...,'PaymentTier','Age', 'ExperienceInCurrentDomain']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pass_through', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when f

**In simple word:***
- The model understands only numbers, not text.
- So the pipeline turns the text column into 0/1 columns (10 columns) and rescales the number columns to a small range (4 columns).
- The 8 orignal columns become 14 columns, which are passed to the model.

**Where the 10 columns come from** 
- Education (3: Bachelors, Masters, PHD) + City (3: Bangalore, New Delhi, Pune) + Gender (2: Female, Male) + EverBenched (2: No, Yes) = 10

**Why 14 Features?**
- The 10 encoded text columns (Categorical) and the 4 scaled number (Numerical) are joined together: 10 + 4 = 14.

In [18]:
# Final pipeline with the Model and Fit

from sklearn.tree import DecisionTreeClassifier

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_stack),
    ('model', DecisionTreeClassifier(max_depth=5, random_state=42)),
])

pipeline.fit(X_train, y_train)
print('Pipeline trained')

Pipeline trained


In [19]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Education','City','Gender',...,'PaymentTier','Age', 'ExperienceInCurrentDomain']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor_stage1', ...), ('preprocessor_stage2', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to th

**Final step (DecisionTreeClassifier):**
- First, all the steps prepare the data (text is turned into numbers, and numbers are scaled).
- At the end, the Decision tree looks at these 14 columns and decides whether the employee will Stay or Leave.
- `max_depth` = 5 means the tree can ask at most 5 question in a eow before it decides.

In [20]:
# Checking Training Accuracy

y_train_pred = pipeline.predict(X_train)

from sklearn.metrics import confusion_matrix

print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("\nClassification Report:\n", classification_report(y_train, y_train_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_train, y_train_pred))


Accuracy: 0.8005427408412483

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.89      0.84      1343
           1       0.80      0.66      0.72       868

    accuracy                           0.80      2211
   macro avg       0.80      0.77      0.78      2211
weighted avg       0.80      0.80      0.80      2211


Confusion Matrix:
 [[1201  142]
 [ 299  569]]


In [21]:
# Model Evaluation

y_pred = pipeline.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Balanced Accuracy:', balanced_accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7884267631103075
Balanced Accuracy: 0.7687960687960689
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       333
           1       0.77      0.67      0.72       220

    accuracy                           0.79       553
   macro avg       0.78      0.77      0.77       553
weighted avg       0.79      0.79      0.79       553



**Insight:**
- Training accuracy is about 80% and test accuracy is about 79%.
- The two scores are very close, so the model is not overfitting. It performs almost the same on data it has never seen.

## Breaking pipeline into Explainable Parts on Test:

In [22]:
# Creating A Test:

my_pred_array = X_test.iloc[15:16]

my_pred_array

,Education,City,Gender,EverBenched,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain
233,Masters,New Delhi,Female,No,2017,2,28,1


In [23]:
pd.DataFrame(preprocessor_stack.transform(my_pred_array), columns=preprocessor_stack[1].get_feature_names_out())

,cat__Education_Bachelors,cat__Education_Masters,cat__Education_PHD,cat__City_Bangalore,cat__City_New Delhi,cat__City_Pune,cat__Gender_Female,cat__Gender_Male,cat__EverBenched_No,cat__EverBenched_Yes,num__JoiningYear,num__PaymentTier,num__Age,num__ExperienceInCurrentDomain
0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.833333,0.5,0.315789,0.142857


In [24]:
my_pred_array.to_json()

'{"Education":{"233":"Masters"},"City":{"233":"New Delhi"},"Gender":{"233":"Female"},"EverBenched":{"233":"No"},"JoiningYear":{"233":2017},"PaymentTier":{"233":2},"Age":{"233":28},"ExperienceInCurrentDomain":{"233":1}}'

In [25]:
# Using Pipeline to do all together (Preprocessing followed by ModelPredict)

# Single Prediction
print('Prediction:', {0: 'Stay', 1: 'Leave'}[pipeline.predict(my_pred_array)[0]])
print('Probability of Leave:', pipeline.predict_proba(my_pred_array)[0][1])

Prediction: Stay
Probability of Leave: 0.18421052631578946


In [26]:
pipeline.predict_proba(my_pred_array)

array([[0.81578947, 0.18421053]])

**Insight:**
- For this employee, the pipeline predicts "Stay" with about 82% confidence (18% chance of leaving).
- The real answer for this employee is also "Stay", so the model predicted correctly.
- `predict` gives the final decision, and `predict_proba` shows how sure the model is.

In [27]:
!pip install dill


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# Save Dictionary to my_feature_dict.pkl

import pickle

with open('my_feature_dict.pkl', 'wb') as fp:
    pickle.dump(my_feature_dict, fp)
    print('dictionary saved successfully to file')

dictionary saved successfully to file


In [29]:
# Save Pipeline to pipeline.pkl

import dill

with open('pipeline.pkl', 'wb') as fp:
    dill.dump(pipeline, fp)
    print('pipeline saved successfully to file')

pipeline saved successfully to file


In [30]:
import dill

#load the saved pipeline from the file

with open('pipeline.pkl', 'rb') as file:
    loaded_pipeline = dill.load(file)
    print('pipeline loaded successfully to file')

pipeline loaded successfully to file


In [31]:
loaded_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Education','City','Gender',...,'PaymentTier','Age', 'ExperienceInCurrentDomain']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor_stage1', ...), ('preprocessor_stage2', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to th

In [32]:
y_pred = loaded_pipeline.predict(my_pred_array)
y_pred

array([0])

`Predicts 0 (Employee is likely to stay, not Churn):`

**Why would i do this step:**

- To make sure our saved model file is working properly and not corrupted.
- To ensure raw data automatically goes through scaling and encoding without crashing.
- To catch and fix any bugs before putting the model into a live web application.